In [1]:
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt

In [2]:
def display_missing_data_info(dataframe, ascending=False):
    missing_values_count = dataframe.isnull().sum()
    missing_values_percentage = (dataframe.isnull().mean() * 100)
    missing_data = pd.DataFrame({
        'Missing Values': missing_values_count,
        'Percentage (%)': missing_values_percentage
    })
    
    missing_data = missing_data[missing_data['Missing Values'] > 0]
    
    sorted_missing_data = missing_data.sort_values(by='Missing Values', ascending=ascending)

    print(sorted_missing_data)
    return sorted_missing_data
    

In [3]:
bureau_balance = pd.read_csv('../data/dseb63_bureau_balance.csv')

In [4]:
bureau_balance

,SK_ID_BUREAU,MONTHS_BALANCE,STATUS
0,5715448,0,C
1,5715448,-1,C
2,5715448,-2,C
3,5715448,-3,C
4,5715448,-4,C
...,...,...,...
27299920,5041336,-47,X
27299921,5041336,-48,X
27299922,5041336,-49,X
27299923,5041336,-50,X


In [5]:
sk_id_bureau = bureau_balance['SK_ID_BUREAU'].unique()

len(sk_id_bureau)

817395

In [6]:
from multiprocessing import Pool, cpu_count

In [7]:
bureau_balance['MAX_MONTHS_BALANCE'] = bureau_balance.groupby('SK_ID_BUREAU')['MONTHS_BALANCE'].transform('max')
bureau_balance['TOTAL_MONTHS'] = bureau_balance.groupby('SK_ID_BUREAU')['MONTHS_BALANCE'].transform('count')

In [8]:
bureau_balance['DIFF'] = bureau_balance['MAX_MONTHS_BALANCE'] - bureau_balance['MONTHS_BALANCE']

In [9]:
bureau_balance_last_12 = bureau_balance[bureau_balance['DIFF'] <= 12]

In [10]:
bureau_balance_last_12['C Last 12 Month'] = np.where(bureau_balance_last_12['STATUS'] == 'C', 1, 0)
bureau_balance_last_12['X Last 12 Month'] = np.where(bureau_balance_last_12['STATUS'] == 'X', 1, 0)
bureau_balance_last_12['0 Last 12 Month'] = np.where(bureau_balance_last_12['STATUS'] == '0', 1, 0)

bureau_balance_last_12['Other Last 12 Month'] = np.where((bureau_balance_last_12['STATUS'] != 'C') & (bureau_balance_last_12['STATUS'] != 'X') & (bureau_balance_last_12['STATUS'] != '0'), 1, 0)

gb_bureau_balance_last_12 = bureau_balance_last_12.groupby('SK_ID_BUREAU').agg({ 'C Last 12 Month': 'sum', 'X Last 12 Month': 'sum', '0 Last 12 Month': 'sum', 'Other Last 12 Month': 'sum' })
gb_bureau_balance_last_12

C:\Users\Admin\AppData\Local\Temp\ipykernel_17540\3286580497.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  bureau_balance_last_12['C Last 12 Month'] = np.where(bureau_balance_last_12['STATUS'] == 'C', 1, 0)
C:\Users\Admin\AppData\Local\Temp\ipykernel_17540\3286580497.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  bureau_balance_last_12['X Last 12 Month'] = np.where(bureau_balance_last_12['STATUS'] == 'X', 1, 0)
C:\Users\Admin\AppData\Local\Temp\ipykernel_17540\3286580497.py:3: SettingWithCopyWarn

,C Last 12 Month,X Last 12 Month,0 Last 12 Month,Other Last 12 Month
SK_ID_BUREAU,,,,
5001709,13,0,0,0
5001710,13,0,0,0
5001711,0,1,3,0
5001712,9,0,4,0
5001713,0,13,0,0
...,...,...,...,...
6842884,13,0,0,0
6842885,0,0,1,12
6842886,13,0,0,0


In [11]:
gb_max_month = bureau_balance.groupby('SK_ID_BUREAU')['MONTHS_BALANCE'].max()
gb_max_month.rename('MAX_MONTHS_BALANCE', inplace=True)
gb_total_months = bureau_balance.groupby('SK_ID_BUREAU')['MONTHS_BALANCE'].count()
gb_total_months.rename('TOTAL_MONTHS', inplace=True)
gb_C = bureau_balance[bureau_balance['STATUS'] == 'C'].groupby('SK_ID_BUREAU')['STATUS'].count()
gb_C.rename('Ratio C', inplace=True)

gb_X = bureau_balance[bureau_balance['STATUS'] == 'X'].groupby('SK_ID_BUREAU')['STATUS'].count()
gb_X.rename('Ratio X', inplace=True)

gb_0 = bureau_balance[bureau_balance['STATUS'] == '0'].groupby('SK_ID_BUREAU')['STATUS'].count()
gb_0.rename('Ratio 0', inplace=True)

gb_Other = bureau_balance[(bureau_balance['STATUS'] != 'C') & (bureau_balance['STATUS'] != 'X') & (bureau_balance['STATUS'] != '0')].groupby('SK_ID_BUREAU')['STATUS'].count()
gb_Other.rename('Ratio Other', inplace=True)

SK_ID_BUREAU
5001718     2
5001720     7
5001722    20
5001757     1
5001786     1
           ..
6842826     1
6842874     1
6842880     2
6842885    12
6842888     1
Name: Ratio Other, Length: 103264, dtype: int64

In [12]:
gb_bureau_balance = pd.merge(gb_bureau_balance_last_12, gb_max_month, on='SK_ID_BUREAU')
gb_bureau_balance = pd.merge(gb_bureau_balance, gb_total_months, on='SK_ID_BUREAU')
gb_bureau_balance = pd.merge(gb_bureau_balance, gb_C, on='SK_ID_BUREAU')
gb_bureau_balance = pd.merge(gb_bureau_balance, gb_X, on='SK_ID_BUREAU')
gb_bureau_balance = pd.merge(gb_bureau_balance, gb_0, on='SK_ID_BUREAU')
gb_bureau_balance = pd.merge(gb_bureau_balance, gb_Other, on='SK_ID_BUREAU')
gb_bureau_balance = gb_bureau_balance.reset_index()

In [13]:
gb_bureau_balance

,SK_ID_BUREAU,C Last 12 Month,X Last 12 Month,0 Last 12 Month,Other Last 12 Month,MAX_MONTHS_BALANCE,TOTAL_MONTHS,Ratio C,Ratio X,Ratio 0,Ratio Other
0,5001718,3,2,6,2,0,39,3,10,24,2
1,5001844,13,0,0,0,0,48,16,12,19,1
2,5001922,13,0,0,0,0,36,19,9,7,1
3,5001981,13,0,0,0,0,67,60,1,3,3
4,5001988,5,2,6,0,0,53,5,15,32,1
...,...,...,...,...,...,...,...,...,...,...,...
20699,6842759,13,0,0,0,0,74,43,9,8,14
20700,6842765,13,0,0,0,0,49,21,9,18,1
20701,6842791,13,0,0,0,0,53,40,8,3,2
20702,6842874,13,0,0,0,0,90,80,1,8,1


In [14]:
gb_bureau_balance['Ratio C'] = gb_bureau_balance['Ratio C'] / gb_bureau_balance['TOTAL_MONTHS']
gb_bureau_balance['Ratio X'] = gb_bureau_balance['Ratio X'] / gb_bureau_balance['TOTAL_MONTHS']
gb_bureau_balance['Ratio 0'] = gb_bureau_balance['Ratio 0'] / gb_bureau_balance['TOTAL_MONTHS']

In [15]:
gb_bureau_balance.to_csv('../data/dseb63_bureau_balance_aggregated.csv', index=False)

### Feature to get

- Duration(mean)
- Total payment
- Last balance
- Continuous
- Count C, 0, X
- % C in last 12 
- % X in last 12

In [5]:
bureau_balance['STATUS'].value_counts()

STATUS
C    13646993
0     7499507
X     5810482
1      242347
5       62406
2       23419
3        8924
4        5847
Name: count, dtype: int64

In [17]:
sample = bureau_balance[bureau_balance['SK_ID_BUREAU'] == 5714717]
sample

,SK_ID_BUREAU,MONTHS_BALANCE,STATUS


Only for active account

In [19]:
appl_train = pd.read_csv('../data/application_train.csv')
bureau = pd.read_csv('../data/bureau.csv')
bureau = pd.merge(bureau[['SK_ID_CURR', 'SK_ID_BUREAU']], appl_train[['SK_ID_CURR', 'TARGET']], on='SK_ID_CURR', how='left')

In [20]:
bureau_balance = pd.merge(bureau_balance, bureau, on='SK_ID_BUREAU', how='left')
bureau_balance

,SK_ID_BUREAU,MONTHS_BALANCE,STATUS,SK_ID_CURR,TARGET
0,5715448,0,C,380361.0,0.0
1,5715448,-1,C,380361.0,0.0
2,5715448,-2,C,380361.0,0.0
3,5715448,-3,C,380361.0,0.0
4,5715448,-4,C,380361.0,0.0
...,...,...,...,...,...
27299920,5041336,-47,X,101874.0,1.0
27299921,5041336,-48,X,101874.0,1.0
27299922,5041336,-49,X,101874.0,1.0
27299923,5041336,-50,X,101874.0,1.0


In [22]:
bureau_balance.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 27299925 entries, 0 to 27299924
Data columns (total 5 columns):
 #   Column          Dtype  
---  ------          -----  
 0   SK_ID_BUREAU    int64  
 1   MONTHS_BALANCE  int64  
 2   STATUS          object 
 3   SK_ID_CURR      float64
 4   TARGET          float64
dtypes: float64(2), int64(2), object(1)
memory usage: 1.0+ GB


In [23]:
bureau_balance_positive = bureau_balance[bureau_balance['TARGET'] == 1]


In [25]:
bureau_balance_positive.groupby('SK_ID_BUREAU').agg({'STATUS': 'first', 'MONTHS_BALANCE': 'max'})

,STATUS,MONTHS_BALANCE
SK_ID_BUREAU,,
5008873,C,0
5008874,C,0
5008875,C,0
5008876,C,0
5008877,C,0
...,...,...
6841952,X,-58
6841953,X,-51
6841954,X,-51
